# Train a Scikit-Learn model in SageMaker and track with MLFlow

## Setup Environment

In [ ]:
!pip install -q --upgrade pip

In [ ]:
!pip uninstall sagemaker -y
!pip install 'sagemaker<3.0'

In [ ]:
!pip install --upgrade scikit-learn

In [ ]:
!pip install numpy==1.26.4

!pip install --upgrade pandas


In [14]:
import sagemaker
import pandas as pd
# from sklearn.datasets import load_boston
from sagemaker.sklearn.estimator import SKLearn
# from sklearn.model_selection import train_test_split

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()

# uri of your remote mlflow server
tracking_uri = 'http://MLflow-MLFLO-JW54W3S07WQL-ddaae86f1ebfd86e.elb.us-east-1.amazonaws.com' 

## Prepare data
We load a dataset from sklearn, split it and send it to S3

In [ ]:
# we use the Boston housing dataset 
# data = load_boston()

# X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.25, random_state=42)

# trainX = pd.DataFrame(X_train, columns=data.feature_names)
# trainX['target'] = y_train

# testX = pd.DataFrame(X_test, columns=data.feature_names)
# testX['target'] = y_test

# trainX.to_csv('boston_train.csv')
# testX.to_csv('boston_test.csv')

In [15]:
# send data to S3. SageMaker will take training data from s3
train_path = sess.upload_data(path='boston_train.csv', bucket=bucket, key_prefix='sagemaker/sklearncontainer')
test_path = sess.upload_data(path='boston_test.csv', bucket=bucket, key_prefix='sagemaker/sklearncontainer')

## Train

In [16]:
hyperparameters = {
    'tracking_uri': tracking_uri,
    'experiment_name': 'boston-housing',
    'n-estimators': 200,
    'min-samples-leaf': 2,
    'features': 'CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT',
    'target': 'target'
}

metric_definitions = [{'Name': 'median-AE', 'Regex': "AE-at-50th-percentile: ([0-9.]+).*$"}]

estimator = SKLearn(
    entry_point='train.py',
    source_dir='source_dir',
    role=role,
    metric_definitions=metric_definitions,
    hyperparameters=hyperparameters,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='1.0-1',
    base_job_name='mlflow',
)

In [17]:
estimator.fit({'train':train_path, 'test': test_path})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: mlflow-2026-06-13-11-15-31-802


2026-06-13 11:15:36 Starting - Starting the training job...
2026-06-13 11:15:52 Starting - Preparing the instances for training...
2026-06-13 11:16:17 Downloading - Downloading input data...
2026-06-13 11:16:42 Downloading - Downloading the training image...
2026-06-13 11:17:28 Training - Training image download completed. Training in progress..2026-06-13 11:17:36,867 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-06-13 11:17:36,871 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-06-13 11:17:36,875 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-06-13 11:17:36,891 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-06-13 11:17:37,196 sagemaker-training-toolkit INFO     Installing module with the following command:
/miniconda3/bin/python -m pip install . -r requirements.txt
Processing /opt/ml/code
  Preparing metadata (setup.py):